In [27]:
# Imports
import json
from lxml import etree

In [28]:
# Helper functions
def get_station_decisions(path) -> set[str]:
    charging_locations = set()
    with open(path, "r") as file:
        data = json.load(file)
    for station_decision in data["station_decisions"]:
        charging_locations.add(station_decision["station_id"])
    return charging_locations

In [29]:
# Parse JSON File
solution_cicerostrasse = "../data_basis/solution_cicerostrasse.json"

solution_muellerstrasse = "../data_basis/solution_muellerstrasse.json"

# Parse xml file
tree = etree.parse("../data_basis/charging_stations.add.xml")
cs_root = tree.getroot()

combined_stations = (
    get_station_decisions(solution_muellerstrasse) 
    | get_station_decisions(solution_cicerostrasse)
)



In [30]:
root = etree.Element(
    "additional",
    nsmap={
        "xsi": "http://www.w3.org/2001/XMLSchema-instance",
    },
)

root.set(
    "{http://www.w3.org/2001/XMLSchema-instance}noNamespaceSchemaLocation",
    "http://sumo.dlr.de/xsd/additional_file.xsd",
)

for chargingStation in cs_root.findall("id"):
    id = chargingStation.get(id)
    if id.partition("_")[2] not in combined_stations:
        cs_root.remove(chargingStation)


# Write XML
tree = etree.ElementTree(cs_root)
tree.write(
    "../preprocessing_output/charging_stations_heuristic.add.xml",
    encoding="utf-8",
    xml_declaration=True,
    pretty_print=True,
)